
# Chapter 7 : Tooling Using external tools


In [66]:
import os

os.environ["GROQ_API_KEY"] = "GROQ_API_KEY"

In [67]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"]
)

In [68]:
model = "openai/gpt-oss-20b"

In [69]:
#!pip install langchain-openai
from langchain_core.tools import tool
from langchain_core.tools import tool as langchain_tool
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
import asyncio
import nest_asyncio

In [70]:
# Create a tool calling agent

if llm:
  """ for the agent's internal steps """
  agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

In [71]:
import yfinance as yf
from langchain_core.tools import tool

@tool
def get_reliance_price() -> str:
    """Get the latest available market data for Reliance Industries Limited (NSE: RELIANCE).
    Use this tool whenever the user asks for Reliance share price, current price,
    latest price, today's price, OHLC, or market data."""

    ticker = yf.Ticker("RELIANCE.NS")
    data = ticker.history(period="1d", interval="1m")

    if data.empty:
        return "Unable to retrieve Reliance Industries price."

    latest = data.iloc[-1]

    return (
        f"Reliance Industries (NSE: RELIANCE) "
        f"latest available price is ₹{latest['Close']:.2f}. "
        f"Open ₹{latest['Open']:.2f}, "
        f"High ₹{latest['High']:.2f}, "
        f"Low ₹{latest['Low']:.2f}, "
        f"Volume {int(latest['Volume'])}."
    )

tools = [get_reliance_price]

In [72]:
tools = [get_reliance_price]

In [73]:
from langchain_openai import ChatOpenAI
import os


agent = create_agent(
    model= llm, # Pass the instantiated ChatOpenAI object
    tools=[get_reliance_price],
    system_prompt="""
You are a financial information assistant.

When the user asks for current or latest Reliance Industries
market data, ALWAYS call get_reliance_price.

When the tool returns market data, report ALL the fields returned
by the tool, including:
- Latest Price
- Open
- High
- Low
- Volume

Do not omit fields.
Do not invent market data.
"""
)

In [74]:
response = await agent.ainvoke({
    "messages": [
        {
            "role": "user",
            "content": "Give me today's Reliance share price with Open, High, Low and Volume."
        }
    ]
})

print(response["messages"][-1].content)

**Reliance Industries (NSE: RELIANCE) – Today**

- **Latest Price:** ₹1,226.00  
- **Open:** ₹1,224.50  
- **High:** ₹1,226.00  
- **Low:** ₹1,224.10  
- **Volume:** 236,840 shares  

*(All figures are from the latest market data returned by the get_reliance_price tool.)*



# --- THE END ---

